# Week 5 — Embeddings
> Goal: Understand how text embeddings work, compare models, and visualize the semantic space of financial documents.

**Topics covered:**
- What is an embedding vector?
- OpenAI `text-embedding-3-small` vs HuggingFace `all-MiniLM-L6-v2`
- Cosine similarity
- Embedding clusters (PCA + UMAP visualization)
- Financial domain: why general embeddings sometimes fail


## 0. Setup

In [ ]:
import sys, os
sys.path.insert(0, '../src')

from dotenv import load_dotenv
load_dotenv('../.env')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

from embedder import get_embedder, cosine_similarity, top_k_similar, compare_models

print('Setup complete.')

---
## 1. What Is an Embedding?

An embedding is a function: **text → float vector** in a high-dimensional space.
Semantically similar text lands near each other geometrically.

```
"Apple revenue declined"  →  [0.021, -0.143, 0.872, ...  ]  (1536 dims)
"Apple profits fell"      →  [0.019, -0.138, 0.869, ...  ]  ← very close!
"Fed raised rates"        →  [0.341,  0.892, -0.12, ...  ]  ← far away
```


In [ ]:
# ── 1.1  Embed a few sentences and inspect the raw vectors ──────────────────
embedder = get_embedder('openai-small')   # 1536-dim

sample_texts = [
    "Apple's quarterly revenue exceeded analyst expectations",
    "Apple beat earnings estimates this quarter",          # semantically close ↑
    "The Federal Reserve raised interest rates by 25 bps",# different topic
    "AAPL stock surged after strong quarterly results",   # related
    "Nvidia reported record data center revenue",          # different company
]

vecs = embedder(sample_texts)
print(f'Shape: {vecs.shape}')           # (5, 1536)
print(f'First vector (first 10 dims):\n  {vecs[0, :10]}')
print(f'L2 norm of first vector: {np.linalg.norm(vecs[0]):.4f}')  # ~1.0 after normalization

---
## 2. Cosine Similarity

For normalized vectors: `cos(θ) = a · b`
- `1.0` = identical
- `0.0` = unrelated
- `-1.0` = opposite (rare in practice)

In [ ]:
# ── 2.1  Pairwise similarity matrix ─────────────────────────────────────────
n = len(sample_texts)
sim_matrix = np.zeros((n, n))

for i in range(n):
    for j in range(n):
        sim_matrix[i, j] = cosine_similarity(vecs[i], vecs[j])

# Visualize
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(sim_matrix, cmap='YlOrRd', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, label='Cosine similarity')

labels = [t[:35] + '...' if len(t) > 35 else t for t in sample_texts]
ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(labels, fontsize=9)

for i in range(n):
    for j in range(n):
        ax.text(j, i, f'{sim_matrix[i,j]:.2f}', ha='center', va='center', fontsize=8)

ax.set_title('Cosine Similarity Matrix — Financial Sentences', fontsize=12, pad=15)
plt.tight_layout()
plt.savefig('plots/similarity_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# What to observe:
# Sentence 0 and 1 should have high similarity (~0.90+)
# Sentence 0 and 2 should have low similarity (~0.30-0.45)
print(f"\nSame topic (0 vs 1): {sim_matrix[0,1]:.4f}  ← should be HIGH")
print(f"Diff topic (0 vs 2): {sim_matrix[0,2]:.4f}  ← should be LOW")
print(f"Related   (0 vs 3): {sim_matrix[0,3]:.4f}  ← should be MEDIUM-HIGH")

---
## 3. Model Comparison: OpenAI vs MiniLM

In [ ]:
# ── 3.1  Side-by-side similarity comparison ──────────────────────────────────
results = compare_models(sample_texts)

# Plot both similarity matrices side by side
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, (model_name, data) in zip(axes, results.items()):
    im = ax.imshow(data['matrix'], cmap='YlOrRd', vmin=0, vmax=1)
    plt.colorbar(im, ax=ax)
    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(labels, fontsize=8)
    ax.set_title(f'{model_name}', fontsize=11)
    for i in range(n):
        for j in range(n):
            ax.text(j, i, f"{data['matrix'][i,j]:.2f}", ha='center', va='center', fontsize=7)

plt.suptitle('Model Comparison: Cosine Similarity on Financial Text', y=1.02, fontsize=13)
plt.tight_layout()
plt.savefig('plots/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 3.2  Benchmark: which model distinguishes financial topics better? ───────
test_pairs = [
    # (text_a, text_b, expected_relationship)
    ("Apple revenue grew 5% year over year",
     "Apple annual sales increased by 5 percent",
     "very_similar"),
    ("Net income margin improved to 25%",
     "Profitability ratio increased to one quarter",
     "similar"),
    ("The company repurchased $10B of shares",
     "Interest rates were raised by the Fed",
     "unrelated"),
    ("EPS beat by $0.15",
     "Earnings per share exceeded estimates",
     "very_similar"),
    ("Gross margin expanded 200 bps",
     "Cost of goods sold declined relative to revenue",
     "similar"),
]

print(f'{'Pair':<45} {'Expected':<15} {'OpenAI':>8} {'MiniLM':>8}')
print('-' * 80)
for ta, tb, expected in test_pairs:
    va_oa = results.get('OpenAI-small', {}).get('vectors')
    va_ml = results.get('MiniLM', {}).get('vectors')
    label = f"{ta[:20]}… / {tb[:20]}…"
    # Re-embed the specific pairs
    oa_emb = get_embedder('openai-small')([ta, tb])
    ml_emb = get_embedder('minilm')([ta, tb])
    oa_sim = cosine_similarity(oa_emb[0], oa_emb[1])
    ml_sim = cosine_similarity(ml_emb[0], ml_emb[1])
    print(f'{label:<45} {expected:<15} {oa_sim:>8.3f} {ml_sim:>8.3f}')

---
## 4. Clustering Financial Sentences

In [ ]:
# ── 4.1  Create a set of labeled financial sentences ─────────────────────────
financial_sentences = {
    'Revenue': [
        "Total net sales increased 8% to $45.2 billion",
        "Revenue from the iPhone segment grew 6% year over year",
        "Services revenue reached a new all-time high of $21 billion",
        "Net sales in Greater China declined 4% versus prior year",
        "Annual revenue for fiscal 2023 was $383 billion",
    ],
    'Risk Factors': [
        "Adverse global economic conditions could impact consumer demand",
        "We face intense competition in all our market segments",
        "Supply chain disruptions may limit our ability to meet demand",
        "Changes in foreign exchange rates affect our international revenue",
        "Regulatory changes in key markets may restrict our operations",
    ],
    'Cash Flow': [
        "Operating cash flow was $110 billion for the fiscal year",
        "Free cash flow increased to $99 billion after capital expenditures",
        "The company returned $90 billion to shareholders through buybacks and dividends",
        "Capital expenditures were $11 billion, primarily for infrastructure",
        "Cash and cash equivalents totaled $62 billion at year end",
    ],
    'Competition': [
        "Samsung remains the largest competitor in the smartphone market",
        "Google's Android platform continues to dominate global market share",
        "Microsoft Azure competes directly with our cloud infrastructure",
        "Competitors have introduced products at lower price points",
        "New entrants from China are increasing competitive pressure",
    ],
    'Margins': [
        "Gross margin expanded 80 basis points to 44.1 percent",
        "Operating margin declined due to increased R&D investment",
        "Product gross margin was 36.6% while Services was 70.8%",
        "Cost of goods sold decreased as a percentage of revenue",
        "EBITDA margin reached 34% for the trailing twelve months",
    ],
}

all_sentences = [s for sentences in financial_sentences.values() for s in sentences]
all_labels = [label for label, sentences in financial_sentences.items() for _ in sentences]

print(f'Total sentences: {len(all_sentences)}')
print(f'Categories: {list(financial_sentences.keys())}')

In [ ]:
# ── 4.2  Embed and reduce to 2D with PCA ─────────────────────────────────────
import os
os.makedirs('plots', exist_ok=True)

embedder = get_embedder('openai-small')
all_vecs = embedder(all_sentences)

pca = PCA(n_components=2, random_state=42)
coords_2d = pca.fit_transform(all_vecs)
print(f'Explained variance ratio: {pca.explained_variance_ratio_}')

# ── 4.3  Plot clusters ────────────────────────────────────────────────────────
categories = list(financial_sentences.keys())
colors = cm.tab10(np.linspace(0, 0.6, len(categories)))
color_map = dict(zip(categories, colors))

fig, ax = plt.subplots(figsize=(10, 8))

for cat in categories:
    idxs = [i for i, l in enumerate(all_labels) if l == cat]
    x = coords_2d[idxs, 0]
    y = coords_2d[idxs, 1]
    ax.scatter(x, y, c=[color_map[cat]], label=cat, s=100, alpha=0.85, edgecolors='white', linewidths=0.5)
    # Add sentence labels (abbreviated)
    for xi, yi, idx in zip(x, y, idxs):
        ax.annotate(all_sentences[idx][:28] + '…', (xi, yi),
                    fontsize=6.5, alpha=0.7, xytext=(4, 4), textcoords='offset points')

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)', fontsize=11)
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)', fontsize=11)
ax.set_title('Financial Sentence Embeddings — PCA (2D)\nOpenAI text-embedding-3-small', fontsize=12)
ax.legend(loc='best', fontsize=9)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig('plots/embedding_clusters_pca.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nObservation: Sentences from the same category should cluster together.')
print('Tight, separated clusters = good embeddings for financial text.')

In [ ]:
# ── 4.4  KMeans clustering: can we recover the original categories? ───────────
n_clusters = len(categories)
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
predicted_clusters = kmeans.fit_predict(all_vecs)

# Map predicted cluster → most common true label (greedy assignment)
from collections import Counter
cluster_to_label = {}
for c in range(n_clusters):
    idxs = [i for i, p in enumerate(predicted_clusters) if p == c]
    most_common = Counter(all_labels[i] for i in idxs).most_common(1)[0][0]
    cluster_to_label[c] = most_common

correct = sum(1 for i, pred in enumerate(predicted_clusters)
              if cluster_to_label[pred] == all_labels[i])
accuracy = correct / len(all_labels)
print(f'Clustering accuracy (KMeans): {accuracy:.1%}  ({correct}/{len(all_labels)} correct)')
print('Higher accuracy = embedding model captures financial topic structure well.')

---
## 5. Nearest Neighbor Search

In [ ]:
# ── 5.1  Simple nearest-neighbor retrieval (RAG building block) ───────────────
query = "What was the company's profitability?"
query_vec = embedder([query])[0]

results = top_k_similar(query_vec, all_vecs, k=5)

print(f'Query: "{query}"')
print(f'{"Rank":<5} {"Score":>6}  Text')
print('-' * 75)
for rank, (idx, score) in enumerate(results, 1):
    print(f'{rank:<5} {score:>6.4f}  [{all_labels[idx]}] {all_sentences[idx]}')

In [ ]:
# ── 5.2  Try multiple financial queries ──────────────────────────────────────
queries = [
    "What are the biggest risks to the business?",
    "How much cash did the company generate?",
    "Who are the main competitors?",
    "What happened to revenue this year?",
]

for q in queries:
    q_vec = embedder([q])[0]
    top1 = top_k_similar(q_vec, all_vecs, k=1)[0]
    idx, score = top1
    print(f'Q: {q}')
    print(f'   → [{all_labels[idx]}, {score:.3f}] {all_sentences[idx]}')
    print()

---
## 6. Financial Domain Challenge

General embeddings sometimes fail on domain-specific jargon.
Let's test with financial abbreviations and ratios.

In [ ]:
# ── 6.1  Does the model understand financial abbreviations? ──────────────────
domain_pairs = [
    ("P/E ratio of 25x",          "Price-to-earnings multiple of 25"),
    ("EV/EBITDA of 15x",          "Enterprise value to EBITDA ratio of 15"),
    ("YoY revenue growth of 8%",   "Year-over-year revenue increase of 8 percent"),
    ("FCF yield of 4.2%",          "Free cash flow yield of 4.2 percent"),
    ("CAGR of 12% over 5 years",   "Compound annual growth rate of 12% over five years"),
    ("bps margin expansion",       "basis points improvement in profit margin"),
]

print(f'{"Abbreviated":<35} {"Expanded":<45} {"Similarity":>10}')
print('-' * 95)
for abbrev, expanded in domain_pairs:
    pair_vecs = embedder([abbrev, expanded])
    sim = cosine_similarity(pair_vecs[0], pair_vecs[1])
    indicator = '✓ good' if sim > 0.85 else ('~ ok' if sim > 0.70 else '✗ poor')
    print(f'{abbrev:<35} {expanded:<45} {sim:>7.3f}  {indicator}')

print("\nNote: Scores below 0.85 suggest the model doesn't fully connect the abbreviation to its meaning.")
print("Solution: Use a domain-specific model (FinBERT) or fine-tune on financial corpora.")

---
## 7. Summary & Key Takeaways

| Finding | Implication |
|---------|-------------|
| OpenAI embeddings have higher similarity on same-topic sentences | Better precision in retrieval |
| MiniLM is fast (local, free) but slightly weaker | Good for development; upgrade for production |
| Financial abbreviations (EV/EBITDA, FCF) may score lower | Consider FinBERT or fine-tuning |
| Clear cluster separation → retrieval will be accurate | Validates our embedding choice |

**Next up → Week 6: Build a FAISS vector database from these embeddings.**

In [ ]:
# ── Save embeddings for use in next notebook ─────────────────────────────────
import json
os.makedirs('../data', exist_ok=True)
np.save('../data/demo_vecs.npy', all_vecs)
with open('../data/demo_sentences.json', 'w') as f:
    json.dump({'sentences': all_sentences, 'labels': all_labels}, f, indent=2)
print('Saved demo_vecs.npy and demo_sentences.json — use these in 02_vectordb.ipynb')